# SCD Data Generator

Interactive notebook for adding, updating, and deleting rows in any bronze table  
to drive SCD1 / SCD2 testing in the AdventureWorks dbt pipeline.

**Workflow**
1. Run **Cell 1** – configure catalog.
2. Run **Cell 2** – imports.
3. Run **Cell 3** – defines the INSERT / UPDATE / DELETE action functions.
4. Run **Cell 4** – choose a table, preview data, then use the action widgets.
5. Trigger your dbt run (`dbt snapshot && dbt run`) and inspect the dimension.


## Cell 1 – Catalog widget


In [0]:
dbutils.widgets.text("catalog", "adventureworks_dev")
catalog_name = dbutils.widgets.get("catalog") or "adventureworks_dev"
print(f"Using catalog: {catalog_name}")

## Cell 2 – Imports & helpers


In [0]:
from datetime import datetime, timezone
import uuid

import ipywidgets as widgets
from IPython.display import display, clear_output
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── table registry (all bronze tables) ─────────────────────────────────
TABLES = {
    # SALES
    "Customer": {"pk": "CustomerID"},
    "SalesOrderHeader": {"pk": "SalesOrderID"},
    "SalesOrderDetail": {"pk": "SalesOrderDetailID"},
    "SalesTerritory": {"pk": "TerritoryID"},
    "SalesTerritoryHistory": {"pk": "BusinessEntityID"},
    "SalesPerson": {"pk": "BusinessEntityID"},
    "SpecialOffer": {"pk": "SpecialOfferID"},
    "SpecialOfferProduct": {"pk": "SpecialOfferID"},
    "ShipMethod": {"pk": "ShipMethodID"},
    "CreditCard": {"pk": "CreditCardID"},
    "Store": {"pk": "BusinessEntityID"},
    # HUMAN RESOURCES
    "Employee": {"pk": "BusinessEntityID"},
    "EmployeePayHistory": {"pk": "BusinessEntityID"},
    "EmployeeDepartmentHistory": {"pk": "BusinessEntityID"},
    # PERSON
    "BusinessEntity": {"pk": "BusinessEntityID"},
    "BusinessEntityAddress": {"pk": "BusinessEntityID"},
    "Person": {"pk": "BusinessEntityID"},
    "Address": {"pk": "AddressID"},
    "StateProvince": {"pk": "StateProvinceID"},
    # PRODUCTION
    "Product": {"pk": "ProductID"},
    "ProductSubcategory": {"pk": "ProductSubcategoryID"},
    "ProductCategory": {"pk": "ProductCategoryID"},
}


def now_ts() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def full_table(table_name: str) -> str:
    return f"{catalog_name}.bronze.{table_name}"


def read_table(table_name: str):
    return spark.table(full_table(table_name))


def get_columns(table_name: str) -> list:
    return read_table(table_name).columns


def get_max_pk(table_name: str) -> int:
    pk = TABLES[table_name]["pk"]
    row = read_table(table_name).agg(F.max(F.col(pk).cast("int"))).collect()[0][0]
    return int(row) if row is not None else 0


print("Helpers loaded. Proceed to Cell 3.")

## Cell 3 – Action implementations

> Run this cell **once** before using the UI in Cell 4 so the action functions are in scope.


In [ ]:
from delta.tables import DeltaTable


def _collect_field_values() -> dict:
    """Return {column: value_or_None} from current field widgets."""
    return {col: (w.value.strip() or None) for col, w in _field_widgets.items()}


# ── INSERT ─────────────────────────────────────────────────────────────────
def _do_insert(table: str, pk: str, ft: str):
    values = _collect_field_values()

    if not values.get(pk):
        next_id = str(get_max_pk(table) + 1)
        values[pk] = next_id
        print(f"ℹ️  Auto-assigned {pk} = {next_id}")

    if "rowguid" in values and not values["rowguid"]:
        values["rowguid"] = str(uuid.uuid4()).upper()

    if "ModifiedDate" in values and not values["ModifiedDate"]:
        values["ModifiedDate"] = now_ts()

    # Build a single-row DataFrame matching the table schema (all strings)
    existing_cols = get_columns(table)
    row = {c: values.get(c) for c in existing_cols}

    df = spark.createDataFrame([row])
    df.write.format("delta").mode("append").saveAsTable(ft)

    print(f"✅ Inserted 1 row into {ft}  ({pk}={values[pk]})")
    print("   ▶ Next: run  dbt snapshot && dbt run  to propagate.")


# ── UPDATE ─────────────────────────────────────────────────────────────────
def _do_update(table: str, pk: str, ft: str):
    pk_value = _pk_filter_widget.value.strip() if _pk_filter_widget else ""
    if not pk_value:
        print(f"❌ Please enter the {pk} of the row to update.")
        return

    values = _collect_field_values()
    # Only update columns that have a non-blank value
    updates = {col: val for col, val in values.items() if val is not None}

    if not updates:
        print("❌ No fields to update — enter at least one new value.")
        return

    # Always bump ModifiedDate
    updates["ModifiedDate"] = now_ts()

    set_clause = ", ".join(f"{c} = '{v}'" for c, v in updates.items())
    sql = f"UPDATE {ft} SET {set_clause} WHERE {pk} = '{pk_value}'"

    spark.sql(sql)

    print(f"✅ Updated {ft}  WHERE {pk}={pk_value}")
    print(f"   Fields changed: {list(updates.keys())}")
    print("   ▶ Next: run  dbt snapshot && dbt run  to propagate.")


# ── DELETE ─────────────────────────────────────────────────────────────────
def _do_delete(table: str, pk: str, ft: str):
    pk_value = _pk_filter_widget.value.strip() if _pk_filter_widget else ""
    if not pk_value:
        print(f"❌ Please enter the {pk} of the row to delete.")
        return

    affected = spark.sql(
        f"SELECT COUNT(*) AS n FROM {ft} WHERE {pk} = '{pk_value}'"
    ).collect()[0]["n"]

    if affected == 0:
        print(f"⚠️  No rows found in {ft} WHERE {pk}={pk_value}")
        return

    spark.sql(f"DELETE FROM {ft} WHERE {pk} = '{pk_value}'")

    print(f"✅ Deleted {affected} row(s) from {ft}  WHERE {pk}={pk_value}")
    print("   ▶ Next: run  dbt snapshot && dbt run  to see hard-delete invalidation.")


print("Action functions ready. Use the UI in Cell 4.")

## Cell 4 – UI: table selector + row editor


In [0]:
# ── layout helpers ─────────────────────────────────────────────────────────
style_label = widgets.Layout(width="200px")
style_input = widgets.Layout(width="340px")
style_wide = widgets.Layout(width="520px")


def label(text):
    return widgets.Label(text, layout=style_label)


# ── top-level controls ─────────────────────────────────────────────────────
w_table = widgets.Dropdown(
    options=sorted(TABLES.keys()),
    value="Customer",
    description="Table:",
    layout=style_wide,
)

w_preview_rows = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description="Preview rows:",
    layout=style_wide,
)

w_preview_btn = widgets.Button(
    description="🔍 Preview table",
    button_style="info",
    layout=widgets.Layout(width="160px"),
)

w_action = widgets.RadioButtons(
    options=["INSERT new row", "UPDATE existing row", "DELETE existing row"],
    value="INSERT new row",
    description="Action:",
    layout=style_wide,
)

# ── dynamic field area (rebuilt when table or action changes) ──────────────
w_fields_box = widgets.VBox([])
out_preview = widgets.Output()
out_status = widgets.Output()

# storage for dynamically-built field widgets (populated in build_fields)
_field_widgets: dict = {}
_pk_filter_widget = None


def build_fields(_=None):
    global _field_widgets, _pk_filter_widget
    table = w_table.value
    action = w_action.value
    columns = get_columns(table)
    pk = TABLES[table]["pk"]

    rows = []

    if action in ("UPDATE existing row", "DELETE existing row"):
        # PK filter to identify the target row
        _pk_filter_widget = widgets.Text(
            placeholder=f"Enter {pk} to target",
            layout=style_input,
        )
        rows.append(widgets.HBox([label(f"🔑 {pk} (target)"), _pk_filter_widget]))
        rows.append(widgets.HTML("<hr style='margin:6px 0'/>"))
    else:
        _pk_filter_widget = None

    _field_widgets = {}

    if action == "DELETE existing row":
        rows.append(widgets.Label("Only the PK value above is needed for DELETE."))
    else:
        # One text input per column
        # For UPDATE, these are the new values; leave blank to keep existing.
        for col in columns:
            hint = "" if action == "INSERT new row" else " (blank = keep existing)"
            auto_hint = ""
            if col == pk and action == "INSERT new row":
                next_id = get_max_pk(table) + 1
                auto_hint = f" (next available: {next_id})"
            if col == "rowguid" and action == "INSERT new row":
                auto_hint = " (auto-generated if blank)"
            if col == "ModifiedDate" and action == "INSERT new row":
                auto_hint = " (auto-set to now if blank)"

            w = widgets.Text(
                placeholder=f"{col}{auto_hint}{hint}",
                layout=style_input,
            )
            _field_widgets[col] = w
            rows.append(widgets.HBox([label(col), w]))

    w_fields_box.children = rows


# ── preview handler ────────────────────────────────────────────────────────
def on_preview(_):
    with out_preview:
        clear_output(wait=True)
        df = read_table(w_table.value).limit(w_preview_rows.value)
        display(df.toPandas())


w_preview_btn.on_click(on_preview)
w_table.observe(build_fields, names="value")
w_action.observe(build_fields, names="value")

# ── execute button ─────────────────────────────────────────────────────────
w_execute_btn = widgets.Button(
    description="▶ Execute",
    button_style="success",
    layout=widgets.Layout(width="140px"),
)


def on_execute(_):
    with out_status:
        clear_output(wait=True)
        table = w_table.value
        action = w_action.value
        pk = TABLES[table]["pk"]
        ft = full_table(table)

        try:
            if action == "INSERT new row":
                _do_insert(table, pk, ft)
            elif action == "UPDATE existing row":
                _do_update(table, pk, ft)
            elif action == "DELETE existing row":
                _do_delete(table, pk, ft)
        except Exception as e:
            print(f"❌ Error: {e}")


w_execute_btn.on_click(on_execute)

# ── initial build ──────────────────────────────────────────────────────────
build_fields()

display(
    widgets.VBox(
        [
            widgets.HBox([w_table, w_preview_rows, w_preview_btn]),
            out_preview,
            widgets.HTML("<b>Action</b>"),
            w_action,
            widgets.HTML("<b>Fields</b>"),
            w_fields_box,
            widgets.HTML("<hr/>"),
            w_execute_btn,
            out_status,
        ]
    )
)

## Cell 5 – SCD validation queries

Run the SQL cells below after triggering your dbt run to verify SCD behaviour.


In [0]:
# ── helper: display snapshot history for any business key ──────────────────
def show_scd_history(snapshot_table: str, bk_col: str, bk_value):
    """
    Print the full SCD2 history for a given business key.

    Parameters
    ----------
    snapshot_table : str
        Fully-qualified gold snapshot table, e.g.
        'adventureworks_dev.gold.snap_product'
    bk_col : str
        Business-key column name, e.g. 'product_bk'
    bk_value : int | str
        Business-key value to inspect
    """
    df = spark.sql(f"""
        SELECT *
        FROM   {snapshot_table}
        WHERE  {bk_col} = {bk_value!r}
        ORDER  BY dbt_valid_from
    """)
    print(f"SCD history — {snapshot_table}  {bk_col}={bk_value}")
    display(df.toPandas())


# ── quick-check: how many open (current) rows exist? ───────────────────────
def show_open_rows(snapshot_table: str):
    df = spark.sql(f"""
        SELECT COUNT(*) AS open_rows
        FROM   {snapshot_table}
        WHERE  dbt_valid_to IS NULL
    """)
    display(df.toPandas())


# ── overlap check: any bk with two open rows? (should always be 0) ─────────
def check_no_overlap(snapshot_table: str, bk_col: str):
    df = spark.sql(f"""
        SELECT {bk_col}, COUNT(*) AS open_versions
        FROM   {snapshot_table}
        WHERE  dbt_valid_to IS NULL
        GROUP  BY {bk_col}
        HAVING COUNT(*) > 1
    """)
    n = df.count()
    if n == 0:
        print(f"✅ No overlapping open rows in {snapshot_table}")
    else:
        print(f"❌ {n} business key(s) have multiple open rows in {snapshot_table}:")
        display(df.toPandas())


# ── example usage (edit and run) ───────────────────────────────────────────
# Note: snap_customer no longer exists — dim_customer is Type 1 (SCD1).
# Use snap_product, snap_employee, or snap_salesterritory below:
#
# show_scd_history(
#     f"{catalog_name}.gold.snap_product",
#     bk_col="product_bk",
#     bk_value=707,
# )
#
# check_no_overlap(f"{catalog_name}.gold.snap_product",        "product_bk")
# check_no_overlap(f"{catalog_name}.gold.snap_employee",       "employee_bk")
# check_no_overlap(f"{catalog_name}.gold.snap_salesterritory", "sales_territory_bk")

print("Validation helpers ready.")